# 05. Classical Machine Learning Benchmarks

Unified training and evaluation pipeline for classical ML baselines using modular `src/`:
- **Models**: XGBoost, Random Forest, Logistic Regression, Linear SVM, KNN.
- **Handling Imbalance**: Automatically computes negative-to-positive ratio (`scale_pos_weight`) and balanced weights.
- **Evaluation**: Computes PR-AUC, ROC-AUC, MCC, and sweeps thresholds (0.05–0.95) to find optimal F1.
- **Logging**: Automatically logs results to centralized `output/experiment_summary.csv`.

In [1]:
import sys
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path so src is importable anywhere
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import clean_tabular_features, get_feature_and_label_matrices, LABEL_COLS
from src.models import train_multi_label_models, predict_multi_label_probabilities
from src.metrics import evaluate_predictions, log_experiment

print(f"✓ Project root: {PROJECT_ROOT}")
print("✓ Successfully imported modules from src (data, models, metrics)")

✓ Project root: /home/emmy/ptmPred
✓ Successfully imported modules from src (data, models, metrics)


In [2]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Feature encoding to use: "onehot", "blosum", "aapc", or "hybrid"
ENCODING_MTHD = "onehot"

# Splitting condition identifier: "random", "group", or "cluster"
SPLIT_CONDITION = "random"

# Models to train: any combination of ["xgboost", "random_forest", "logistic_regression", "svm", "knn"]
MODELS_TO_RUN = ["xgboost", "random_forest", "logistic_regression", "svm", "knn"]

# Data paths (fully portable across local machine, lab server, and Colab)
DATA_DIR = PROJECT_ROOT / f"data_engineered/{ENCODING_MTHD}"
TRAIN_CSV = DATA_DIR / f"train_with_features_{ENCODING_MTHD}_split.csv"
VAL_CSV = DATA_DIR / f"val_with_features_{ENCODING_MTHD}_split.csv"

OUTPUT_DIR = PROJECT_ROOT / "output"
SUMMARY_CSV = OUTPUT_DIR / "experiment_summary.csv"

print("Configuration:")
print(f"  Encoding:        {ENCODING_MTHD}")
print(f"  Split Condition: {SPLIT_CONDITION}")
print(f"  Models:          {MODELS_TO_RUN}")
print(f"  Train Path:      {TRAIN_CSV}")
print(f"  Val Path:        {VAL_CSV}")
print(f"  Summary Ledger:  {SUMMARY_CSV}")

Configuration:
  Encoding:        onehot
  Split Condition: random
  Models:          ['xgboost', 'random_forest', 'logistic_regression', 'svm', 'knn']
  Train Path:      /home/emmy/ptmPred/data_engineered/onehot/train_with_features_onehot_split.csv
  Val Path:        /home/emmy/ptmPred/data_engineered/onehot/val_with_features_onehot_split.csv
  Summary Ledger:  /home/emmy/ptmPred/output/experiment_summary.csv


In [3]:
# Load and clean datasets using src.data
if TRAIN_CSV.exists() and VAL_CSV.exists():
    raw_train = pd.read_csv(TRAIN_CSV)
    raw_val = pd.read_csv(VAL_CSV)
    
    clean_train = clean_tabular_features(raw_train, drop_constants=True, encode_categoricals=False)
    clean_val = clean_tabular_features(raw_val, drop_constants=True, encode_categoricals=False)
    
    X_train, y_train, feature_cols = get_feature_and_label_matrices(clean_train)
    X_val, y_val, _ = get_feature_and_label_matrices(clean_val)
    
    # Ensure train and val have identical columns
    common_cols = [c for c in feature_cols if c in X_val.columns]
    X_train = X_train[common_cols]
    X_val = X_val[common_cols]
    
    print(f"✓ Data loaded & cleaned:")
    print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
else:
    raise FileNotFoundError(f"Split files not found:\n  {TRAIN_CSV}\n  {VAL_CSV}")

✓ Data loaded & cleaned:
  X_train: (71208, 109), y_train: (71208, 3)
  X_val:   (17802, 109), y_val:   (17802, 3)


In [4]:
# Train and evaluate selected models
run_records = []

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*70}\nBENCHMARKING: {model_name.upper()} ({ENCODING_MTHD} | {SPLIT_CONDITION})\n{'='*70}")
    
    # 1. Train multi-label models using src.models
    trained_models = train_multi_label_models(model_name, X_train, y_train, LABEL_COLS)
    
    # 2. Predict probabilities using src.models
    probs_df = predict_multi_label_probabilities(trained_models, X_val, LABEL_COLS)
    
    # 3. Evaluate each label using src.metrics
    for label in LABEL_COLS:
        metrics = evaluate_predictions(y_val[label].values, probs_df[label].values)
        
        record = {
            "run_id": f"{model_name}_{ENCODING_MTHD}_{SPLIT_CONDITION}",
            "split_condition": SPLIT_CONDITION,
            "model": model_name,
            "encoding": ENCODING_MTHD,
            "label": label,
            **metrics
        }
        run_records.append(record)
        
        # Log directly into centralized experiment_summary.csv
        log_experiment(str(SUMMARY_CSV), record)
        
        print(f"  {label:<22}: PR-AUC={metrics['pr_auc']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | F1={metrics['f1_optimal']:.4f} (thresh={metrics['optimal_threshold']})")

print(f"\n✓ All models evaluated and logged to {SUMMARY_CSV}")


BENCHMARKING: XGBOOST (onehot | random)


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:cys_ctx_-1_Y: str, cys_ctx_1_B: str

In [ ]:
# View summary of current run
current_df = pd.DataFrame(run_records)
macro_df = current_df.groupby(['model', 'encoding', 'split_condition'])[['pr_auc', 'roc_auc', 'f1_optimal', 'mcc_optimal']].mean().reset_index()

print("\n" + "="*70 + "\nMACRO-AVERAGE PERFORMANCE SUMMARY\n" + "="*70)
print(macro_df.to_string(index=False))

# Plot side-by-side comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.barplot(data=macro_df, x='model', y='pr_auc', palette='viridis')
plt.title(f"Macro PR-AUC ({ENCODING_MTHD} | {SPLIT_CONDITION})")
plt.xticks(rotation=20)
plt.ylabel("Macro PR-AUC")

plt.subplot(1, 2, 2)
sns.barplot(data=macro_df, x='model', y='f1_optimal', palette='magma')
plt.title(f"Macro F1 at Optimal Threshold ({ENCODING_MTHD} | {SPLIT_CONDITION})")
plt.xticks(rotation=20)
plt.ylabel("Macro F1")
plt.tight_layout()
plt.show()